In [1]:
!pip install -q "transformers==4.51.2" "accelerate==1.7.0" decord ffmpeg-python imageio opencv-python nibabel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.1/362.1 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 49.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
from google.colab import drive, userdata
from huggingface_hub import login
drive.mount("/content/drive")
login(userdata.get("HF_TOKEN"))

Mounted at /content/drive


In [3]:
PROMPT_ID = "caption_v1"
PROMPT = (
    "Write a figure caption for this medical image as it would appear in a radiology "
    "case report. State the imaging modality and anatomical region, and describe visible "
    "findings. Do not speculate about the diagnosis. Respond in one or two sentences."
)
MAX_NEW_TOKENS = 100
N_IMAGES = None

MODEL_NAME = "Hulu-Med-4B"

DRIVE = "/content/drive/MyDrive/vlm-caption-bench"
SUBSET_FILE = f"{DRIVE}/subset_2000.csv"
OUT_DIR = f"{DRIVE}/predictions"

In [4]:
import os, zipfile
import pandas as pd
from pathlib import Path

os.makedirs(OUT_DIR, exist_ok=True)
if not os.path.isdir("test"):
    with zipfile.ZipFile(f"{DRIVE}/test.zip") as z:
        z.extractall("test")

ids = pd.read_csv(SUBSET_FILE)["ID"].astype(str).tolist()
if N_IMAGES:
    ids = ids[:N_IMAGES]
by_stem = {p.stem: p for p in Path("test").rglob("*.jpg")}
missing = [i for i in ids if i not in by_stem]
assert not missing, f"немає файлів для {len(missing)} ID, напр. {missing[:3]}"
paths = [by_stem[i] for i in ids]
print(len(paths), "images")

2000 images


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor

model_id = "ZJU-AI4H/Hulu-Med-4B"
dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    model_id, trust_remote_code=True, torch_dtype=dtype, device_map="auto")
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
tokenizer = processor.tokenizer

config.json: 0.00B [00:00, ?B/s]

configuration_hulumed_qwen3.py: 0.00B [00:00, ?B/s]

configuration_hulumed_encoder.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ZJU-AI4H/Hulu-Med-4B:
- configuration_hulumed_encoder.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_hulumed_encoder.py: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/786 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

In [6]:
def caption(image_path, prompt=PROMPT):
    conversation = [{"role": "user", "content": [
        {"type": "image", "image": {"image_path": str(image_path)}},
        {"type": "text", "text": prompt},
    ]}]
    inputs = processor(conversation=conversation, add_system_prompt=True,
                       add_generation_prompt=True, return_tensors="pt")
    inputs = {k: v.to(model.device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}
    if "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].to(dtype)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             temperature=None, top_p=None, top_k=None,
                             pad_token_id=tokenizer.eos_token_id)
    return processor.batch_decode(out, skip_special_tokens=True, use_think=False)[0].strip()

In [7]:
for p in paths[:3]:
    print(p.name, "->", caption(p), "\n")

ROCOv2_2023_test_005382.jpg -> Coronal T2-weighted MRI of the right thigh demonstrates a hyperintense, well-defined fluid collection within the subcutaneous tissue. The lesion appears to be encapsulated and does not show any signs of infiltration into surrounding structures. 

ROCOv2_2023_test_004808.jpg -> Radiograph of the right upper thoracic region reveals multiple well-defined, round radiopaque structures within the soft tissues. These lesions are consistent with foreign bodies or calcifications and appear to be located in the subcutaneous tissue near the shoulder area. 

ROCOv2_2023_test_002741.jpg -> Doppler ultrasound of the right testicle demonstrates normal vascular flow with a resistive index of 0.50 and a systolic/diastolic ratio of 2.00. The spectral Doppler waveform shows a peak systolic velocity of 7.2 cm/s and an end-diastolic velocity of 3.6 cm/s, consistent with normal testicular perfusion. 



In [8]:
import json, gc

OUT = Path(OUT_DIR) / f"{MODEL_NAME}__{PROMPT_ID}.jsonl"

def read_done(path):
    ids, good = set(), []
    if path.exists():
        for l in path.open(encoding="utf-8"):
            try:
                r = json.loads(l)
            except json.JSONDecodeError:
                continue
            ids.add(r["image_id"])
            good.append(l if l.endswith("\n") else l + "\n")
        path.write_text("".join(good), encoding="utf-8")
    return ids

done = read_done(OUT)
print(len(paths), "images,", len(done), "already done")

failed = []
with OUT.open("a", encoding="utf-8") as f:
    for i, p in enumerate(paths, 1):
        image_id = p.stem
        if image_id in done:
            continue

        cap, oom = None, False
        try:
            cap = caption(p)
        except torch.cuda.OutOfMemoryError:
            oom = True

        if oom:
            gc.collect()
            torch.cuda.empty_cache()
            print(f"OOM on {p.name} - пропущено, буде повторено при наступному запуску")
            failed.append(p.name)
            continue

        rec = {
            "image_id": image_id,
            "caption": cap,
            "model": MODEL_NAME,
            "prompt_id": PROMPT_ID,
            "max_new_tokens": MAX_NEW_TOKENS,
            "do_sample": False
        }
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        f.flush()
        os.fsync(f.fileno())
        torch.cuda.empty_cache()
        if i % 50 == 0:
            print(f"{i}/{len(paths)} done")
print("failed:", failed)

2000 images, 2000 already done
failed: []


In [ ]:
import numpy as np
failed_paths = [p for p in paths if p.stem not in read_done(OUT)]
sz = lambda p: max(Image.open(p).size)
print("failed:", sorted(sz(p) for p in failed_paths))
print("median all:", np.median([sz(p) for p in paths[:200]]))

failed: [964, 1125, 1151, 1153, 1163, 1175, 1182, 1183, 1187, 1188, 1190, 1192, 1193, 1194, 1202, 1207, 1213, 1216, 1222, 1224, 1227, 1236, 1252, 1266, 1303, 1313, 1318, 1330, 1333, 1337, 1363, 1378, 1399, 1404, 1404, 1417, 1423, 1434, 1471, 1473, 1479, 1482, 1542, 1554, 1592, 1635, 1690, 1824, 1905, 1944, 2003, 2066, 2076, 2268, 2322, 2403, 2812, 3164]
median all: 750.0


In [10]:
import gc, tempfile
from PIL import Image

def caption_fallback(p, sides=(1536, 1280, 1024, 768, 512)):
    for s in sides:
        tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False); tmp.close()
        cap, oom = None, False
        try:
            img = Image.open(p).convert("RGB")
            if max(img.size) > s:
                img.thumbnail((s, s))
            img.save(tmp.name)
            cap = caption(tmp.name)
        except torch.cuda.OutOfMemoryError:
            oom = True
        finally:
            os.unlink(tmp.name)
        if not oom:
            return cap, s
        gc.collect(); torch.cuda.empty_cache()
    return None, None

done = read_done(OUT)
todo = [p for p in paths if p.stem not in done]
print(len(todo), "залишилось")

still_failed = []
with OUT.open("a", encoding="utf-8") as f:
    for p in todo:
        cap, s = caption_fallback(p)
        if cap is None:
            still_failed.append(p.name)
            continue
        rec = {"image_id": p.stem, "caption": cap, "model": MODEL_NAME,
               "prompt_id": PROMPT_ID, "max_new_tokens": MAX_NEW_TOKENS,
               "do_sample": False, "max_image_side": s}
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        f.flush(); os.fsync(f.fileno())
print("failed:", still_failed)

0 залишилось
failed: []
